In [1]:
from snowflake.snowpark import Session
from config import SNOWFLAKE_CONN_PARAMS

# Create the session 
session = Session.builder.configs(SNOWFLAKE_CONN_PARAMS).create()

# Defined variables for the respective SQL queries needed to access the tables
crimeDataQuery = """
select * from CRIME_ETL_DB.CLEAN.CRIME_DATA_CLEAN
"""

populationQuery = """
select FORCE_NAME, YEAR, TOTAL_POPULATION from CRIME_ETL_DB.CLEAN.POPULATION_CLEAN
"""

walesDeprivation = """
select * from CRIME_ETL_DB.CLEAN.DEPRIVATION_WALES_CLEAN
"""

englandDeprivation = """
select * from CRIME_ETL_DB.CLEAN.DEPRIVATION_ENGLAND_CLEAN
"""

# Selecting the crime data with the respective SQL and coverting the output to a dataframe
crimeRun = session.sql(crimeDataQuery)
crimeDf = crimeRun.to_pandas()

# Selecting the population data with the respective SQL and coverting the output to a dataframe
populationRun = session.sql(populationQuery)
populationDf = populationRun.to_pandas()

# Selecting the Welsh deprivation data with the respective SQL and coverting the output to a dataframe
walesRun = session.sql(walesDeprivation)
walesDepDf = walesRun.to_pandas()

# Selecting the English deprivation data with the respective SQL and coverting the output to a dataframe
englandRun = session.sql(englandDeprivation)
englandDepDf = englandRun.to_pandas()



In [2]:
from feature_engineering_transformation import featureTransformation

# Applying the feature transformation layer to the primary crime datasets as well as the enrichment deprivation and population data
combinedData = featureTransformation(crimeDf, populationDf, walesDepDf, englandDepDf)

In [3]:
from feature_engineering_transformation import aggregationExport

# Applying the combined aggregation and export layer to the combined data from the previous layer
final = aggregationExport(combinedData)

Validation - Duplicate records at reporting grain: 0

Final Missing Value Summary:
FORCE         0
YEAR          0
QUARTER       0
DISTRICT      0
CRIME_TYPE    0
dtype: int64

--- Final Dataset Summary Statistics ---
Total Grain Rows: 8250
Unique Police Forces Covered: 4 (['Metropolitan' 'South Wales' 'Sussex' 'West Midlands'])
Temporal Window Span: 2024 to 2026

Data Sucessfully converted into CSV file.
